# Анализ файла `Выгрузка_CRM_все_клиенты.csv`

Шаги:
1. Чтение файла с пропуском первых 33 строк (SQL-скрипт).
2. Сохранение подготовленной (чистой) выгрузки.
3. Расчет уникальных ИНН.
4. Разбивка уникальных ИНН по ЗО, ГСЗ, ГСК, ГК.

In [ ]:
import os
import pandas as pd

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
SRC_FILE = f"{DATA_DIR}/Выгрузка_CRM_все_клиенты.csv"
CLEAN_FILE = f"{DATA_DIR}/Выгрузка_CRM_все_клиенты_clean.csv"

SKIP_ROWS = 33  # данные начинаются с 34-й строки
SEP = ";"

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print("SRC_FILE:", SRC_FILE)
print("CLEAN_FILE:", CLEAN_FILE)

In [ ]:
if not os.path.exists(SRC_FILE):
    raise FileNotFoundError(f"Не найден файл: {SRC_FILE}")


def read_semicolon_keep_all(path, skip_rows=33, sep=";"):
    """Читает CSV после skip_rows без потери строк.

    Если в строке встречается лишний ';', склеивает лишние токены в колонку COMMENT.
    """
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    if len(lines) <= skip_rows:
        raise ValueError(f"Недостаточно строк после skip_rows={skip_rows}: {path}")

    header = lines[skip_rows].rstrip("\n")
    columns = header.split(sep)
    ncols = len(columns)

    # По скриншоту SQL ожидаем поле comment.
    # На всякий случай ищем несколько вариантов имени колонки.
    comment_candidates = ["comment", "COMMENT", "COMMENT_TEXT"]
    comment_idx = None
    for c in comment_candidates:
        if c in columns:
            comment_idx = columns.index(c)
            break

    records = []
    fixed_more = 0
    fixed_less = 0

    for raw_line in lines[skip_rows + 1 :]:
        raw = raw_line.rstrip("\n")
        parts = raw.split(sep)

        if len(parts) == ncols:
            records.append(parts)
            continue

        if len(parts) > ncols and comment_idx is not None:
            extra = len(parts) - ncols
            merged_comment = sep.join(parts[comment_idx : comment_idx + extra + 1])
            repaired = parts[:comment_idx] + [merged_comment] + parts[comment_idx + extra + 1 :]
            if len(repaired) == ncols:
                records.append(repaired)
                fixed_more += 1
                continue

        if len(parts) < ncols:
            records.append(parts + [""] * (ncols - len(parts)))
            fixed_less += 1
            continue

        # fallback: излишек приклеиваем в последний столбец
        tail = sep.join(parts[ncols - 1 :])
        repaired = parts[: ncols - 1] + [tail]
        records.append(repaired)
        fixed_more += 1

    df_out = pd.DataFrame(records, columns=columns)
    return df_out, ncols, fixed_more, fixed_less


# Чтение без потери строк
# Важно: не используем low_memory с engine=python (несовместимо).
df, expected_cols, fixed_more, fixed_less = read_semicolon_keep_all(
    SRC_FILE, skip_rows=SKIP_ROWS, sep=SEP
)

raw_rows = len(df)
raw_cols = df.shape[1]
print(f"Загружено: {raw_rows:,} строк, {raw_cols} колонок (ожидалось {expected_cols})")
print(f"Исправлено строк с лишними ';': {fixed_more:,}")
print(f"Дополнено коротких строк: {fixed_less:,}")
display(df.head(3))

# Нормализация базовых полей
if "X_INN" not in df.columns:
    raise KeyError("В файле отсутствует колонка X_INN")
df["inn"] = df["X_INN"].apply(normalize_inn)

if "X_AREA_RESP" in df.columns:
    df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
    df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)
else:
    df["segment"] = None

df.to_csv(CLEAN_FILE, index=False, encoding="utf-8")
print(f"Подготовленная выгрузка сохранена: {CLEAN_FILE}")

In [ ]:
# 1) Общее число уникальных ИНН
unique_inn_total = df["inn"].dropna().nunique()
print(f"Уникальных ИНН: {unique_inn_total:,}")

In [ ]:
# 2) Разбивка уникальных ИНН по ЗО (укрупненный сегмент как в analysis_crm_segments.ipynb)
# + отдельная категория для немаппированных значений.

work_seg = df.dropna(subset=["inn"]).copy()

# Базовая таблица только по 3 целевым сегментам
zo_inn = (
    work_seg.groupby("segment")["inn"]
    .nunique()
    .rename("Уникальных ИНН")
    .reset_index()
    .sort_values("Уникальных ИНН", ascending=False)
)
print("Разбивка уникальных ИНН по ЗО (segment):")
display(zo_inn)

# Расширенная таблица: добавляем категорию Прочие/немаппированные ЗО
work_seg["segment_with_other"] = work_seg["segment"].fillna("Прочие/немаппированные ЗО")
zo_inn_with_other = (
    work_seg.groupby("segment_with_other")["inn"]
    .nunique()
    .rename("Уникальных ИНН")
    .reset_index()
    .rename(columns={"segment_with_other": "segment"})
    .sort_values("Уникальных ИНН", ascending=False)
)
print("Разбивка уникальных ИНН по ЗО (с учетом Прочие/немаппированные ЗО):")
display(zo_inn_with_other)

if "X_AREA_RESP" in df.columns:
    zo_raw = (
        work_seg.groupby("X_AREA_RESP")["inn"]
        .nunique()
        .rename("Уникальных ИНН")
        .reset_index()
        .sort_values("Уникальных ИНН", ascending=False)
    )
    print("Исходные значения X_AREA_RESP (для контроля):")
    display(zo_raw)

In [ ]:
# 3) Разбивка уникальных ИНН по ГСЗ (NAME_1), ГСК (NAME_2), ГК (NAME_3)
# + отдельные таблицы заполненности (Пусто vs Заполнено).

group_tables = {}
fill_tables = {}

for col, title in [
    ("NAME_1", "ГСЗ"),
    ("NAME_2", "ГСК"),
    ("NAME_3", "ГК"),
]:
    if col not in df.columns:
        print(f"Колонка {col} отсутствует, блок '{title}' пропущен")
        continue

    base = df.dropna(subset=["inn"])[["inn", col]].copy()
    base[col] = base[col].astype(str).str.strip()

    # Пустые значения приводим к явному маркеру для наглядности таблиц.
    base.loc[base[col].isin(["", "nan", "None"]), col] = "[ПУСТО]"

    t = (
        base.groupby(col)["inn"]
        .nunique()
        .rename("Уникальных ИНН")
        .reset_index()
        .sort_values("Уникальных ИНН", ascending=False)
    )
    group_tables[col] = t

    is_filled = base[col] != "[ПУСТО]"
    fill = pd.DataFrame([
        {"Статус": "Заполнено", "Уникальных ИНН": base.loc[is_filled, "inn"].nunique()},
        {"Статус": "Пусто", "Уникальных ИНН": base.loc[~is_filled, "inn"].nunique()},
    ])
    fill_tables[col] = fill

    print(f"Разбивка уникальных ИНН по {title} ({col}):")
    display(t)

    print(f"Заполненность {title} ({col}):")
    display(fill)

In [ ]:
# 4) Экспорт итогов в единый Excel
# Листы: summary, zo, gsz, gsk, gk

OUT_XLSX = f"{DATA_DIR}/crm_all_clients_summary.xlsx"

summary = pd.DataFrame([
    {"Метрика": "Строк в очищенной выгрузке", "Значение": len(df)},
    {"Метрика": "Колонок в очищенной выгрузке", "Значение": df.shape[1]},
    {"Метрика": "Уникальных ИНН", "Значение": int(df["inn"].dropna().nunique())},
])

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    summary.to_excel(w, sheet_name="summary", index=False)

    # ЗО: берем расширенную таблицу с категорией Прочие/немаппированные ЗО
    zo_inn_with_other.to_excel(w, sheet_name="zo", index=False)

    # ГСЗ / ГСК / ГК
    if "NAME_1" in group_tables:
        group_tables["NAME_1"].to_excel(w, sheet_name="gsz", index=False)
    if "NAME_2" in group_tables:
        group_tables["NAME_2"].to_excel(w, sheet_name="gsk", index=False)
    if "NAME_3" in group_tables:
        group_tables["NAME_3"].to_excel(w, sheet_name="gk", index=False)

print(f"Экспорт завершен: {OUT_XLSX}")